<div style="text-align:left;">
    <span style="
        display:inline-block;
        background-color:#4169E1;
        color:white;
        padding:10px 20px;
        border-radius:8px;
        font-size:45px;
        font-weight:bold;
    ">
        XGBoost Bias Analysis
    </span>
</div>

**Author:** Sarra Chouk 

**Student ID:** 60300372

**Project:** EHR Mortality Risk Prediction  

**Dataset:** MIMIC-IV

**Date:** April 4, 2026  

---

### **Objective**
To evaluate whether the trained model performs differently across demographic groups.

### **Setup and Library Imports**

In [16]:
import warnings
warnings.filterwarnings("ignore")

import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from plotly.subplots import make_subplots
import plotly.graph_objects as go

from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

mpl.rcParams["font.family"] = ["Arial"]
mpl.rcParams["font.size"] = 12
mpl.rcParams["axes.titlesize"] = 14
mpl.rcParams["axes.labelsize"] = 12
mpl.rcParams["xtick.labelsize"] = 11
mpl.rcParams["ytick.labelsize"] = 11

### **Define Paths**

In [17]:
MODEL_ARTIFACTS_DIR = Path("../artifacts/xgboost")
BIAS_OUTPUT_DIR = MODEL_ARTIFACTS_DIR / "bias_analysis_outputs"
BIAS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Model artifacts dir:", MODEL_ARTIFACTS_DIR.resolve())
print("Bias output dir:", BIAS_OUTPUT_DIR.resolve())

Model artifacts dir: /Users/sarrachouk/Desktop/ehr-mortality-prediction/artifacts/xgboost
Bias output dir: /Users/sarrachouk/Desktop/ehr-mortality-prediction/artifacts/xgboost/bias_analysis_outputs


### **Load Saved Model and Prediction Artifacts**

In [18]:
model = joblib.load(MODEL_ARTIFACTS_DIR / "xgb_model.joblib")
metadata = json.load(open(MODEL_ARTIFACTS_DIR / "metadata.json", "r", encoding="utf-8"))

best_threshold = metadata.get("best_threshold", 0.5)

test_predictions = pd.read_csv(MODEL_ARTIFACTS_DIR / "test_predictions.csv")
deployment_predictions = pd.read_csv(MODEL_ARTIFACTS_DIR / "deployment_predictions.csv")

print("Best threshold from metadata:", best_threshold)
print("Test predictions shape:", test_predictions.shape)
print("Deployment predictions shape:", deployment_predictions.shape)

Best threshold from metadata: 0.84
Test predictions shape: (83356, 3)
Deployment predictions shape: (82211, 3)


### **Load Saved Feature Datasets**

In [19]:
X_test_features = pd.read_csv(MODEL_ARTIFACTS_DIR / "X_test_used.csv")
X_deployment_features = pd.read_csv(MODEL_ARTIFACTS_DIR / "X_deployment_used.csv")

print("X_test_features shape:", X_test_features.shape)
print("X_deployment_features shape:", X_deployment_features.shape)

X_test_features shape: (83356, 49)
X_deployment_features shape: (82211, 49)


### **Identify Demographic Feature Columns**

In [20]:
GENDER_COL = "gender"
AGE_COL = "age"

race_cols = [c for c in X_test_features.columns if c.lower().startswith("race_")]

print("Gender column:", GENDER_COL)
print("Age column:", AGE_COL)
print("Detected race one-hot columns:")
print(race_cols)

if GENDER_COL not in X_test_features.columns:
    raise ValueError(f"{GENDER_COL} not found in X_test_features")

if AGE_COL not in X_test_features.columns:
    raise ValueError(f"{AGE_COL} not found in X_test_features")

if len(race_cols) == 0:
    raise ValueError("No one-hot encoded race columns were detected.")

Gender column: gender
Age column: age
Detected race one-hot columns:
['race_black', 'race_asian', 'race_hispanic_latino', 'race_other']


### **Build Fairness Analysis Tables**

In [21]:
test_bias_df = X_test_features.copy().reset_index(drop=True)
deployment_bias_df = X_deployment_features.copy().reset_index(drop=True)

test_bias_df["y_true"] = test_predictions["y_true"].values
test_bias_df["y_proba"] = test_predictions["y_proba"].values
test_bias_df["y_pred"] = test_predictions["y_pred"].values

deployment_bias_df["y_true"] = deployment_predictions["y_true"].values
deployment_bias_df["y_proba"] = deployment_predictions["y_proba"].values
deployment_bias_df["y_pred"] = deployment_predictions["y_pred"].values

print("test_bias_df shape:", test_bias_df.shape)
print("deployment_bias_df shape:", deployment_bias_df.shape)

test_bias_df shape: (83356, 52)
deployment_bias_df shape: (82211, 52)


### **Build and Reconstruct Age and Race Groups**

In [22]:
def reconstruct_race_group(df, race_columns):
    df = df.copy()

    race_matrix = df[race_columns]

    max_col = race_matrix.idxmax(axis=1)

    all_zero_mask = (race_matrix.sum(axis=1) == 0)

    df["race_group"] = max_col.str.replace(r"^race_", "", regex=True)
    df.loc[all_zero_mask, "race_group"] = "Unknown"

    return df

test_bias_df = reconstruct_race_group(test_bias_df, race_cols)
deployment_bias_df = reconstruct_race_group(deployment_bias_df, race_cols)

print("Test race groups:")
print(test_bias_df["race_group"].value_counts(dropna=False))

print("\nDeployment race groups:")
print(deployment_bias_df["race_group"].value_counts(dropna=False))

def build_age_groups(df, age_col):
    df = df.copy()
    df["age_group"] = pd.cut(
        df[age_col],
        bins=[-np.inf, 39, 59, np.inf],
        labels=["<40", "40-59", "60+"]
    )
    return df

test_bias_df = build_age_groups(test_bias_df, AGE_COL)
deployment_bias_df = build_age_groups(deployment_bias_df, AGE_COL)

print("\nTest age groups:")
print(test_bias_df["age_group"].value_counts(dropna=False))

test_bias_df[GENDER_COL] = test_bias_df[GENDER_COL].astype(str).str.strip().str.title()
deployment_bias_df[GENDER_COL] = deployment_bias_df[GENDER_COL].astype(str).str.strip().str.title()

test_bias_df["race_group"] = test_bias_df["race_group"].astype(str).str.strip().str.title()
deployment_bias_df["race_group"] = deployment_bias_df["race_group"].astype(str).str.strip().str.title()

print("\nGender groups in test:")
print(test_bias_df[GENDER_COL].value_counts(dropna=False))

print("\nRace groups in test:")
print(test_bias_df["race_group"].value_counts(dropna=False))

Test race groups:
race_group
Unknown            54784
black              13781
other               6914
hispanic_latino     4891
asian               2986
Name: count, dtype: int64

Deployment race groups:
race_group
Unknown            54411
black              13203
other               6684
hispanic_latino     4851
asian               3062
Name: count, dtype: int64

Test age groups:
age_group
60+      44118
40-59    23397
<40      15841
Name: count, dtype: int64

Gender groups in test:
gender
0    43387
1    39969
Name: count, dtype: int64

Race groups in test:
race_group
Unknown            54784
Black              13781
Other               6914
Hispanic_Latino     4891
Asian               2986
Name: count, dtype: int64


### **Define Fairness Evaluation Metrics**

In [23]:
def safe_roc_auc(y_true, y_score):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_score)

def safe_pr_auc(y_true, y_score):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return average_precision_score(y_true, y_score)

def compute_group_metrics(df, group_col):
    rows = []

    for group_value, group_df in df.groupby(group_col, dropna=False):
        y_true = group_df["y_true"].values
        y_pred = group_df["y_pred"].values
        y_proba = group_df["y_proba"].values

        rows.append({
            "group_column": group_col,
            "group_value": str(group_value),
            "n_samples": int(len(group_df)),
            "positive_rate": float(np.mean(y_true)),
            "predicted_positive_rate": float(np.mean(y_pred)),
            "accuracy": float(accuracy_score(y_true, y_pred)),
            "precision": float(precision_score(y_true, y_pred, zero_division=0)),
            "recall": float(recall_score(y_true, y_pred, zero_division=0)),
            "f1": float(f1_score(y_true, y_pred, zero_division=0)),
            "roc_auc": float(safe_roc_auc(y_true, y_proba)) if not np.isnan(safe_roc_auc(y_true, y_proba)) else np.nan,
            "pr_auc": float(safe_pr_auc(y_true, y_proba)) if not np.isnan(safe_pr_auc(y_true, y_proba)) else np.nan,
            "tp": int(((y_true == 1) & (y_pred == 1)).sum()),
            "fp": int(((y_true == 0) & (y_pred == 1)).sum()),
            "tn": int(((y_true == 0) & (y_pred == 0)).sum()),
            "fn": int(((y_true == 1) & (y_pred == 0)).sum()),
        })

    return pd.DataFrame(rows).sort_values("n_samples", ascending=False).reset_index(drop=True)

### **Compute Fairness Metrics on the Test Set**

In [24]:
test_gender_metrics = compute_group_metrics(test_bias_df, GENDER_COL)
test_age_metrics = compute_group_metrics(test_bias_df, "age_group")
test_race_metrics = compute_group_metrics(test_bias_df, "race_group")

print("Test fairness metrics by gender")
display(test_gender_metrics)

print("Test fairness metrics by age group")
display(test_age_metrics)

print("Test fairness metrics by race")
display(test_race_metrics)

Test fairness metrics by gender


,group_column,group_value,n_samples,positive_rate,predicted_positive_rate,accuracy,precision,recall,f1,roc_auc,pr_auc,tp,fp,tn,fn
0,gender,0,43387,0.019268,0.026506,0.964598,0.195652,0.269139,0.226586,0.869066,0.156393,225,925,41626,611
1,gender,1,39969,0.023368,0.037504,0.953539,0.192128,0.308351,0.236745,0.861434,0.173332,288,1211,37824,646


Test fairness metrics by age group


,group_column,group_value,n_samples,positive_rate,predicted_positive_rate,accuracy,precision,recall,f1,roc_auc,pr_auc,tp,fp,tn,fn
0,age_group,60+,44118,0.032776,0.050909,0.936171,0.195013,0.302905,0.237270,0.812413,0.173069,438,1808,40864,1008
1,age_group,40-59,23397,0.011540,0.014318,0.979784,0.197015,0.244444,0.218182,0.881372,0.139675,66,269,22858,204
2,age_group,<40,15841,0.003409,0.004293,0.993435,0.132353,0.166667,0.147541,0.952989,0.088499,9,59,15728,45


Test fairness metrics by race


,group_column,group_value,n_samples,positive_rate,predicted_positive_rate,accuracy,precision,recall,f1,roc_auc,pr_auc,tp,fp,tn,fn
0,race_group,Unknown,54784,0.020681,0.024551,0.963018,0.168030,0.199470,0.182405,0.843837,0.132675,226,1119,52532,907
1,race_group,Black,13781,0.011973,0.008635,0.982875,0.201681,0.145455,0.169014,0.871143,0.120330,24,95,13521,141
2,race_group,Other,6914,0.056407,0.152155,0.864044,0.238593,0.643590,0.348128,0.875163,0.283276,251,801,5723,139
3,race_group,Hispanic_Latino,4891,0.006338,0.007360,0.988346,0.138889,0.161290,0.149254,0.879404,0.114112,5,31,4829,26
4,race_group,Asian,2986,0.017080,0.032485,0.955124,0.072165,0.137255,0.094595,0.838581,0.097218,7,90,2845,44


### **Compute Fairness Metrics on the Deployment Set**

In [25]:
deployment_gender_metrics = compute_group_metrics(deployment_bias_df, GENDER_COL)
deployment_age_metrics = compute_group_metrics(deployment_bias_df, "age_group")
deployment_race_metrics = compute_group_metrics(deployment_bias_df, "race_group")

print("Deployment fairness metrics by gender")
display(deployment_gender_metrics)

print("Deployment fairness metrics by age group")
display(deployment_age_metrics)

print("Deployment fairness metrics by race")
display(deployment_race_metrics)

Deployment fairness metrics by gender


,group_column,group_value,n_samples,positive_rate,predicted_positive_rate,accuracy,precision,recall,f1,roc_auc,pr_auc,tp,fp,tn,fn
0,gender,0,42777,0.019637,0.025294,0.965308,0.202403,0.260714,0.227888,0.872888,0.169191,219,863,41074,621
1,gender,1,39434,0.023609,0.037176,0.953213,0.188267,0.296455,0.230288,0.859449,0.161150,276,1190,37313,655


Deployment fairness metrics by age group


,group_column,group_value,n_samples,positive_rate,predicted_positive_rate,accuracy,precision,recall,f1,roc_auc,pr_auc,tp,fp,tn,fn
0,age_group,60+,43514,0.032656,0.049547,0.936871,0.192486,0.292048,0.232038,0.814762,0.165765,415,1741,40352,1006
1,age_group,40-59,23209,0.012107,0.013874,0.979189,0.186335,0.213523,0.199005,0.892529,0.151647,60,262,22666,221
2,age_group,<40,15488,0.004455,0.004520,0.993608,0.285714,0.289855,0.287770,0.933141,0.188594,20,50,15369,49


Deployment fairness metrics by race


,group_column,group_value,n_samples,positive_rate,predicted_positive_rate,accuracy,precision,recall,f1,roc_auc,pr_auc,tp,fp,tn,fn
0,race_group,Unknown,54411,0.020602,0.024168,0.963169,0.164259,0.192685,0.177340,0.847119,0.119589,216,1099,52191,905
1,race_group,Black,13203,0.014088,0.009619,0.980989,0.244094,0.166667,0.198083,0.870849,0.136435,31,96,12921,155
2,race_group,Other,6684,0.051765,0.149162,0.867893,0.230692,0.664740,0.342517,0.880332,0.313847,230,767,5571,116
3,race_group,Hispanic_Latino,4851,0.010719,0.007833,0.983509,0.131579,0.096154,0.111111,0.882764,0.088003,5,33,4766,47
4,race_group,Asian,3062,0.021555,0.023187,0.963749,0.183099,0.196970,0.189781,0.895406,0.176944,13,58,2938,53


### **Save Fairness Tables**

In [26]:
test_gender_metrics.to_csv(BIAS_OUTPUT_DIR / "test_gender_metrics.csv", index=False)
test_age_metrics.to_csv(BIAS_OUTPUT_DIR / "test_age_metrics.csv", index=False)
test_race_metrics.to_csv(BIAS_OUTPUT_DIR / "test_race_metrics.csv", index=False)

deployment_gender_metrics.to_csv(BIAS_OUTPUT_DIR / "deployment_gender_metrics.csv", index=False)
deployment_age_metrics.to_csv(BIAS_OUTPUT_DIR / "deployment_age_metrics.csv", index=False)
deployment_race_metrics.to_csv(BIAS_OUTPUT_DIR / "deployment_race_metrics.csv", index=False)

print("Saved fairness metric tables.")

Saved fairness metric tables.


### **Compute Disparity Summaries**

In [27]:
def compute_disparity_summary(metrics_df, metric_cols):
    summary_rows = []

    for metric in metric_cols:
        values = metrics_df[metric].dropna()
        if len(values) == 0:
            continue

        summary_rows.append({
            "group_column": metrics_df["group_column"].iloc[0],
            "metric": metric,
            "min_value": float(values.min()),
            "max_value": float(values.max()),
            "absolute_gap": float(values.max() - values.min())
        })

    return pd.DataFrame(summary_rows)

metric_cols_for_gap = [
    "positive_rate",
    "predicted_positive_rate",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc"
]

test_gender_gap = compute_disparity_summary(test_gender_metrics, metric_cols_for_gap)
test_age_gap = compute_disparity_summary(test_age_metrics, metric_cols_for_gap)
test_race_gap = compute_disparity_summary(test_race_metrics, metric_cols_for_gap)

deployment_gender_gap = compute_disparity_summary(deployment_gender_metrics, metric_cols_for_gap)
deployment_age_gap = compute_disparity_summary(deployment_age_metrics, metric_cols_for_gap)
deployment_race_gap = compute_disparity_summary(deployment_race_metrics, metric_cols_for_gap)

print("Test gender disparity summary")
display(test_gender_gap)

print("Test age disparity summary")
display(test_age_gap)

print("Test race disparity summary")
display(test_race_gap)

Test gender disparity summary


,group_column,metric,min_value,max_value,absolute_gap
0,gender,positive_rate,0.019268,0.023368,0.004100
1,gender,predicted_positive_rate,0.026506,0.037504,0.010998
2,gender,precision,0.192128,0.195652,0.003524
3,gender,recall,0.269139,0.308351,0.039212
4,gender,f1,0.226586,0.236745,0.010159
5,gender,roc_auc,0.861434,0.869066,0.007632
6,gender,pr_auc,0.156393,0.173332,0.016940


Test age disparity summary


,group_column,metric,min_value,max_value,absolute_gap
0,age_group,positive_rate,0.003409,0.032776,0.029367
1,age_group,predicted_positive_rate,0.004293,0.050909,0.046616
2,age_group,precision,0.132353,0.197015,0.064662
3,age_group,recall,0.166667,0.302905,0.136238
4,age_group,f1,0.147541,0.237270,0.089729
5,age_group,roc_auc,0.812413,0.952989,0.140576
6,age_group,pr_auc,0.088499,0.173069,0.084570


Test race disparity summary


,group_column,metric,min_value,max_value,absolute_gap
0,race_group,positive_rate,0.006338,0.056407,0.050069
1,race_group,predicted_positive_rate,0.007360,0.152155,0.144795
2,race_group,precision,0.072165,0.238593,0.166428
3,race_group,recall,0.137255,0.643590,0.506335
4,race_group,f1,0.094595,0.348128,0.253533
5,race_group,roc_auc,0.838581,0.879404,0.040823
6,race_group,pr_auc,0.097218,0.283276,0.186059


### **Fairness Analysis Plots**

In [28]:
def relabel_gender(df, gender_col):
    df = df.copy()

    gender_map = {
        0: "Female",
        1: "Male",
        "0": "Female",
        "1": "Male"
    }

    df[gender_col] = df[gender_col].map(gender_map).fillna(df[gender_col].astype(str))
    return df

test_bias_df = relabel_gender(test_bias_df, GENDER_COL)
deployment_bias_df = relabel_gender(deployment_bias_df, GENDER_COL)

print(test_bias_df[GENDER_COL].value_counts(dropna=False))
print(deployment_bias_df[GENDER_COL].value_counts(dropna=False))

test_gender_metrics = compute_group_metrics(test_bias_df, GENDER_COL)
test_age_metrics = compute_group_metrics(test_bias_df, "age_group")
test_race_metrics = compute_group_metrics(test_bias_df, "race_group")

deployment_gender_metrics = compute_group_metrics(deployment_bias_df, GENDER_COL)
deployment_age_metrics = compute_group_metrics(deployment_bias_df, "age_group")
deployment_race_metrics = compute_group_metrics(deployment_bias_df, "race_group")

gender
Female    43387
Male      39969
Name: count, dtype: int64
gender
Female    42777
Male      39434
Name: count, dtype: int64


In [29]:
plotly_template = "plotly_white"
COLOR_BLUE = "#3146D3"
COLOR_PINK = "#EE3158"
COLOR_ORANGE = "#FFA800"

common_layout = dict(
    template=plotly_template,
    font=dict(family="Arial", size=13, color="#1F2937"),
    title_font=dict(size=20, color="#111827"),
    legend_title_text="Outcome",
    margin=dict(l=50, r=40, t=80, b=50),
    paper_bgcolor="white",
    plot_bgcolor="white"
)

print("Shared chart styling configured.")


def plot_fairness_subplots_plotly(
    gender_df,
    age_df,
    race_df,
    metric="pr_auc",
    title="Fairness Comparison Across Groups",
    filename="fairness_subplots.html"
):
    gender_df = gender_df.sort_values(metric, ascending=False)
    age_df = age_df.sort_values(metric, ascending=False)
    race_df = race_df.sort_values(metric, ascending=False)

    metric_label = (
        metric.replace("_", " ").upper()
        if metric in ["pr_auc", "roc_auc"]
        else metric.replace("_", " ").title()
    )

    fig = make_subplots(
        rows=1,
        cols=3,
        subplot_titles=("Gender", "Age Group", "Race Group"),
        horizontal_spacing=0.08
    )

    fig.add_trace(
        go.Bar(
            x=gender_df["group_value"],
            y=gender_df[metric],
            name="Gender",
            marker_color=COLOR_BLUE,
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Bar(
            x=age_df["group_value"],
            y=age_df[metric],
            name="Age Group",
            marker_color=COLOR_PINK,
        ),
        row=1, col=2
    )

    fig.add_trace(
        go.Bar(
            x=race_df["group_value"],
            y=race_df[metric],
            name="Race Group",
            marker_color=COLOR_ORANGE,
        ),
        row=1, col=3
    )

    fig.update_layout(
        title=dict(
            text=f"<b>{title}</b>",
            x=0.0,
            xanchor="left"
        ),
        showlegend=False,
        height=500,
        width=1200,
        **common_layout
    )

    fig.update_yaxes(title_text=metric_label, row=1, col=1)
    fig.update_yaxes(title_text="", row=1, col=2)
    fig.update_yaxes(title_text="", row=1, col=3)

    fig.update_xaxes(tickangle=0, row=1, col=1)   
    fig.update_xaxes(tickangle=0, row=1, col=2)   
    fig.update_xaxes(tickangle=30, row=1, col=3)  

    fig.write_html(BIAS_OUTPUT_DIR / filename)
    fig.show()

Shared chart styling configured.


In [30]:
plot_fairness_subplots_plotly(
    test_gender_metrics,
    test_age_metrics,
    test_race_metrics,
    metric="pr_auc",
    title="             Test Set PR-AUC Across Gender, Age, and Race Groups",
    filename="test_pr_auc_fairness_subplots.html"
)

In [31]:
plot_fairness_subplots_plotly(
    test_gender_metrics,
    test_age_metrics,
    test_race_metrics,
    metric="recall",
    title="             Test Set Recall Across Gender, Age, and Race Groups",
    filename="test_recall_fairness_subplots.html"
)

In [32]:
plot_fairness_subplots_plotly(
    deployment_gender_metrics,
    deployment_age_metrics,
    deployment_race_metrics,
    metric="pr_auc",
    title="             Deployment Set PR-AUC Across Gender, Age, and Race Groups",
    filename="deployment_pr_auc_fairness_subplots.html"
)

In [33]:
plot_fairness_subplots_plotly(
    deployment_gender_metrics,
    deployment_age_metrics,
    deployment_race_metrics,
    metric="recall",
    title="             Deployment Set Recall Across Gender, Age, and Race Groups",
    filename="deployment_recall_fairness_subplots.html"
)